# SCM `condition_occurrence` — text mapping handoff

Loads row-level output from `_exponent.results_store.omop_mapping_scm_condition_final_output_v1` (produced by `OMOP_mapping_notebook_script_scm_condition.py`) into `omop_silver` / `omop_mapping` / `omop_scm`.

## Load policy (AUTO vs REVIEW)
- **AUTO_MATCH only** (`INCLUDE_REVIEW_REQUIRED = False`): high-confidence slice (mapping default ≥ **0.88** after widen pass).
- **Include REVIEW_REQUIRED** (`True`, **default**): also loads moderate matches (≥ **0.52**). Expect more volume and more noise — use `omop_mapping_scm_condition_review_v1` to spot-check.

## Run order
1. Refresh mapping results (mapping script).
2. Run **this** notebook.
3. `allscripts_scm_condition_era.ipynb` → `allscripts_scm_episode.ipynb` → `allscripts_scm_episode_event.ipynb`.

## Warning
`allscripts_sunrise_condition_occurrence.ipynb` performs a **full delete** of all `allscripts_scm` condition rows in silver/mapping and **truncates** gold before its coded-ICD path. If you rely on **this** text path, do **not** run that notebook’s destructive cells afterward unless it has been changed to preserve text-derived keys (`sxacd_text_condition` in `condition_occurrence_source_value`).

In [ ]:
from pyspark.sql import functions as F

_sep = chr(31)
source = "allscripts_scm"
MAPPING_FINAL = "_exponent.results_store.omop_mapping_scm_condition_final_output_v1"

# False = AUTO_MATCH only. True = also load REVIEW_REQUIRED (wider funnel).
INCLUDE_REVIEW_REQUIRED = True

statuses = ["AUTO_MATCH"]
if INCLUDE_REVIEW_REQUIRED:
    statuses.append("REVIEW_REQUIRED")

print(f"Mapping table: {MAPPING_FINAL}")
print(f"Loading final_status in: {statuses}")

In [ ]:
df_map = spark.table(MAPPING_FINAL).filter(F.col("final_status").isin(statuses))

df_map = df_map.filter(
    F.col("omop_concept_id").isNotNull() & (F.col("omop_concept_id") > 0)
    & F.col("client_guid").isNotNull()
    & F.col("source_id").isNotNull()
)

df_silver = df_map.select(
    F.col("omop_concept_id").cast("int").alias("condition_concept_id"),
    F.coalesce(
        F.to_date(F.col("authored_dtm")),
        F.to_date(F.current_date()),
    ).alias("condition_start_date"),
    F.col("authored_dtm").cast("timestamp").alias("condition_start_datetime"),
    F.lit(None).cast("date").alias("condition_end_date"),
    F.lit(None).cast("timestamp").alias("condition_end_datetime"),
    F.lit(32817).alias("condition_type_concept_id"),
    F.lit(0).alias("condition_status_concept_id"),
    F.lit(None).cast("string").alias("stop_reason"),
    F.substring(F.col("source_value"), 1, 50).alias("condition_source_value"),
    F.lit(0).alias("condition_source_concept_id"),
    F.lit(None).cast("string").alias("condition_status_source_value"),
    F.concat_ws(
        _sep, F.lit(source), F.lit("cv3client"), F.lit("GUID"), F.col("client_guid").cast("string")
    ).alias("person_source_value"),
    F.when(
        F.col("client_visit_guid").isNotNull(),
        F.concat(F.lit(f"{source} | "), F.col("client_visit_guid").cast("string")),
    ).otherwise(F.lit(None).cast("string")).alias("visit_occurrence_source_value"),
    F.concat_ws(
        _sep,
        F.lit(source),
        F.lit("sxacd_text_condition"),
        F.lit("source_id"),
        F.col("source_id").cast("string"),
    ).alias("condition_occurrence_source_value"),
    F.lit(source).alias("source_system"),
)

# Resolve person_id (omop_silver.condition_occurrence uses IDs, not *_source_value)
stp = spark.table("_exponent.omop_mapping.source_to_person").filter(
    F.col("active_flag") == F.lit(True)
)
df_silver = df_silver.join(
    stp.select("person_id", "person_source_value"),
    on="person_source_value",
    how="inner",
).drop("person_source_value")

# Optional visit resolution (same key pattern as other SCM notebooks)
stvo = spark.table("_exponent.omop_mapping.source_to_visit_occurrence").filter(
    (F.col("source_system") == F.lit(source)) & (F.col("active_flag") == F.lit(True))
)
df_silver = df_silver.join(
    stvo.select(
        F.col("visit_occurrence_source_value"),
        F.col("visit_occurrence_id").alias("_visit_occurrence_id"),
    ),
    on="visit_occurrence_source_value",
    how="left",
).withColumn("visit_occurrence_id", F.col("_visit_occurrence_id")).drop(
    "_visit_occurrence_id"
)

df_silver = df_silver.select(
    "condition_occurrence_source_value",
    "person_id",
    "condition_concept_id",
    "condition_start_date",
    "condition_start_datetime",
    "condition_end_date",
    "condition_end_datetime",
    "condition_type_concept_id",
    "condition_status_concept_id",
    "stop_reason",
    F.lit(None).cast("long").alias("provider_id"),
    "visit_occurrence_id",
    F.lit(None).cast("long").alias("visit_detail_id"),
    "visit_occurrence_source_value",
    "condition_source_value",
    "condition_source_concept_id",
    "condition_status_source_value",
    "source_system",
)

df_silver = df_silver.dropDuplicates(["condition_occurrence_source_value"])

n = df_silver.count()
print(f"Silver staging rows (after person join): {n:,}")
# Preview is capped at 25 rows; the count above is the full batch.
display(df_silver.limit(25))

df_silver.createOrReplaceTempView("silver_condition_occurrence_text")

In [ ]:
_silver_merge = spark.sql("""
MERGE INTO _exponent.omop_silver.condition_occurrence AS t
USING silver_condition_occurrence_text AS s
ON t.condition_occurrence_source_value = s.condition_occurrence_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.condition_concept_id <=> s.condition_concept_id)
  OR NOT (t.condition_start_date <=> s.condition_start_date)
  OR NOT (t.condition_start_datetime <=> s.condition_start_datetime)
  OR NOT (t.condition_end_date <=> s.condition_end_date)
  OR NOT (t.condition_end_datetime <=> s.condition_end_datetime)
  OR NOT (t.condition_type_concept_id <=> s.condition_type_concept_id)
  OR NOT (t.condition_status_concept_id <=> s.condition_status_concept_id)
  OR NOT (t.stop_reason <=> s.stop_reason)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.visit_occurrence_id <=> s.visit_occurrence_id)
  OR NOT (t.visit_detail_id <=> s.visit_detail_id)
  OR NOT (t.visit_occurrence_source_value <=> s.visit_occurrence_source_value)
  OR NOT (t.condition_source_value <=> s.condition_source_value)
  OR NOT (t.condition_source_concept_id <=> s.condition_source_concept_id)
  OR NOT (t.condition_status_source_value <=> s.condition_status_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id                     = s.person_id,
  t.condition_concept_id          = s.condition_concept_id,
  t.condition_start_date          = s.condition_start_date,
  t.condition_start_datetime      = s.condition_start_datetime,
  t.condition_end_date            = s.condition_end_date,
  t.condition_end_datetime        = s.condition_end_datetime,
  t.condition_type_concept_id     = s.condition_type_concept_id,
  t.condition_status_concept_id   = s.condition_status_concept_id,
  t.stop_reason                   = s.stop_reason,
  t.provider_id                   = s.provider_id,
  t.visit_occurrence_id           = s.visit_occurrence_id,
  t.visit_detail_id               = s.visit_detail_id,
  t.visit_occurrence_source_value = s.visit_occurrence_source_value,
  t.condition_source_value        = s.condition_source_value,
  t.condition_source_concept_id   = s.condition_source_concept_id,
  t.condition_status_source_value = s.condition_status_source_value,
  t.source_system                 = s.source_system,
  t.last_mod_tsp                  = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
  condition_occurrence_source_value,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  visit_occurrence_source_value,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.condition_occurrence_source_value,
  s.person_id,
  s.condition_concept_id,
  s.condition_start_date,
  s.condition_start_datetime,
  s.condition_end_date,
  s.condition_end_datetime,
  s.condition_type_concept_id,
  s.condition_status_concept_id,
  s.stop_reason,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.visit_occurrence_source_value,
  s.condition_source_value,
  s.condition_source_concept_id,
  s.condition_status_source_value,
  s.source_system,
  current_timestamp()
);
""")
print("Silver MERGE operation metrics (one row):")
display(_silver_merge)

In [ ]:
spark.sql("""
INSERT INTO _exponent.omop_mapping.source_to_condition_occurrence (
    source_system,
    condition_occurrence_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.condition_occurrence_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    current_timestamp() AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, condition_occurrence_source_value
    FROM _exponent.omop_silver.condition_occurrence
    WHERE source_system = 'allscripts_scm'
      AND condition_occurrence_source_value LIKE '%sxacd_text_condition%'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_condition_occurrence x
  ON s.condition_occurrence_source_value = x.condition_occurrence_source_value
 AND x.source_system = 'allscripts_scm';
""")

In [ ]:
_gold_merge = spark.sql("""
MERGE INTO _exponent.omop_scm.condition_occurrence AS gold
USING (
  SELECT
    sco.condition_occurrence_id,
    s.person_id,
    s.condition_concept_id,
    s.condition_start_date,
    s.condition_start_datetime,
    s.condition_end_date,
    s.condition_end_datetime,
    s.condition_type_concept_id,
    s.condition_status_concept_id,
    s.stop_reason,
    s.provider_id AS provider_id,
    s.visit_occurrence_id AS visit_occurrence_id,
    s.visit_detail_id AS visit_detail_id,
    s.condition_source_value,
    s.condition_source_concept_id,
    s.condition_status_source_value
  FROM _exponent.omop_silver.condition_occurrence s
  JOIN _exponent.omop_mapping.source_to_condition_occurrence sco
    ON sco.condition_occurrence_source_value = s.condition_occurrence_source_value
   AND sco.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_id = s.person_id
   AND stp.active_flag = TRUE
  WHERE s.source_system = 'allscripts_scm'
    AND s.condition_occurrence_source_value LIKE '%sxacd_text_condition%'
) AS src
ON gold.condition_occurrence_id = src.condition_occurrence_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                     = src.person_id,
  gold.condition_concept_id          = src.condition_concept_id,
  gold.condition_start_date          = src.condition_start_date,
  gold.condition_start_datetime      = src.condition_start_datetime,
  gold.condition_end_date            = src.condition_end_date,
  gold.condition_end_datetime        = src.condition_end_datetime,
  gold.condition_type_concept_id     = src.condition_type_concept_id,
  gold.condition_status_concept_id   = src.condition_status_concept_id,
  gold.stop_reason                   = src.stop_reason,
  gold.provider_id                   = src.provider_id,
  gold.visit_occurrence_id           = src.visit_occurrence_id,
  gold.visit_detail_id               = src.visit_detail_id,
  gold.condition_source_value        = src.condition_source_value,
  gold.condition_source_concept_id   = src.condition_source_concept_id,
  gold.condition_status_source_value = src.condition_status_source_value

WHEN NOT MATCHED THEN INSERT (
  condition_occurrence_id,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value
) VALUES (
  src.condition_occurrence_id,
  src.person_id,
  src.condition_concept_id,
  src.condition_start_date,
  src.condition_start_datetime,
  src.condition_end_date,
  src.condition_end_datetime,
  src.condition_type_concept_id,
  src.condition_status_concept_id,
  src.stop_reason,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.condition_source_value,
  src.condition_source_concept_id,
  src.condition_status_source_value
);
""")
print("Gold MERGE operation metrics (one row):")
display(_gold_merge)